# Multiclass Perceptron 

## 1. Problem Setup

Assume we have a dataset

$$
\{(x_i,y_i)\}_{i=1}^{N}
$$

where

$$
x_i \in \mathbb{R}^D
$$

is the feature vector and

$$
y_i \in \{1,2,\dots,K\}
$$

is the class label.

The goal is to classify each input into one of $K$ classes.

---

## 2. Linear Model

For multiclass classification, we learn a **weight vector for each class**.

$$
W =
\begin{bmatrix}
w_1^T \\
w_2^T \\
\vdots \\
w_K^T
\end{bmatrix}
\in \mathbb{R}^{K \times D}
$$

The score for class $k$ is

$$
s_k = w_k^T x
$$

---

## 3. Adding the Bias Term

To include an intercept, we augment the feature vector

$$
x =
\begin{bmatrix}
1 \\
x
\end{bmatrix}
$$

The weight matrix becomes

$$
W \in \mathbb{R}^{K \times (D+1)}
$$

Thus the score becomes

$$
s_k = w_k^T x
$$

---

## 4. Prediction Rule

The predicted class is the one with the highest score

$$
\hat{y} = \arg\max_{k} \; w_k^T x
$$

This is called a **winner-takes-all rule**.

---

## 5. One-Hot Encoding

The labels are mapped to indices

$$
y_i \rightarrow \{0,1,\dots,K-1\}
$$

This allows indexing of the corresponding class weights.

---

## 6. Objective Function

The perceptron does not explicitly minimize a smooth loss.

Instead, it minimizes classification errors using the rule:

$$
\text{if } \hat{y}_i \neq y_i
$$

then update the weights.

---

## 7. Update Rule

For a misclassified sample $(x_i, y_i)$:

- Let $y_t$ be the true class  
- Let $y_f$ be the predicted (false) class  

The weights are updated as

$$
w_{y_t} = w_{y_t} + \alpha x_i
$$

$$
w_{y_f} = w_{y_f} - \alpha x_i
$$

where $\alpha$ is the learning rate.

---

## 8. Matrix Form Update

For multiple misclassified samples, updates can be written as

$$
W[y_t] \; += \; \alpha X_m
$$

$$
W[y_f] \; -= \; \alpha X_m
$$

where

$$
X_m
$$

contains misclassified inputs.

---

## 9. Initialization

The weights are initialized as

$$
W = 0
$$

At this point all class scores are equal.

---

## 10. Iterative Optimization

For each epoch:

Compute scores

$$
S = XW^T
$$

Compute predictions

$$
\hat{y} = \arg\max S
$$

Find misclassified samples

$$
\{i : \hat{y}_i \neq y_i\}
$$

Update weights using the perceptron rule.

---

## 11. Convergence Criterion

The algorithm stops when

$$
\hat{y}_i = y_i \quad \forall i
$$

i.e., no misclassifications remain.

This means the data is **linearly separable**.

---

## 12. Geometric Interpretation

Each class has a linear score

$$
w_k^T x
$$

The decision boundary between class $i$ and $j$ is

$$
(w_i - w_j)^T x = 0
$$

Thus the model creates **linear decision regions** in feature space.

---

## 13. Prediction

After training, predictions are computed using

$$
\hat{y} = \arg\max_{k} w_k^T x
$$

---

## 14. Algorithm Summary

Initialize

$$
W = 0
$$

Repeat for each epoch:

Compute scores

$$
S = XW^T
$$

Compute predictions

$$
\hat{y} = \arg\max S
$$

For each misclassified sample:

$$
w_{y_t} = w_{y_t} + \alpha x
$$

$$
w_{y_f} = w_{y_f} - \alpha x
$$

Stop if no misclassifications remain.

---

## 15. Final Optimization Perspective

The perceptron implicitly solves

$$
\text{find } W
$$
such that

$$y_i = \arg\max_k w_k^T x_i$$

for all training samples.

This produces a classifier that separates classes using **linear decision boundaries**.

In [1]:
class MulticlassPerceptron:
    def __init__(self,epochs=2,alpha=0.001, add_bias =False):
        self.epochs = epochs
        self.alpha = alpha
        self.add_bias = add_bias

        self.weights = None

    def _one_hot_encode(self,y):
        unique_classes , encoded_classes = np.unique(y,return_inverse=True) 
        num_classes = len(unique_classes)

        return unique_classes , encoded_classes , num_classes

    def fit(self,X,y):
        X = np.asarray(X)
        
        y = np.asarray(y)
        y = y.reshape(-1)


        if X.ndim==1:
            X = X.reshape(-1,1)

            
        N , D = X.shape

        if self.add_bias :
            X = np.column_stack((np.ones(N),X))
            D+=1


        self.unique_classes , y_encoded, self.num_classes  = self._one_hot_encode(y)
  
        self.weights = np.zeros((self.num_classes,D))


        for i in range(self.epochs):
            scores = X @ self.weights.T
            
            
            y_hat = np.argmax(scores,axis=1)

            misclassified_indexes = np.where(y_hat != y_encoded)[0]


            if len(misclassified_indexes) >0:

                X_m = X[misclassified_indexes]
                y_t = y_encoded[misclassified_indexes]
                y_f = y_hat[misclassified_indexes]
          
                np.add.at(self.weights, y_t, self.alpha * X_m)
                np.add.at(self.weights, y_f, -self.alpha * X_m)

            if len(misclassified_indexes)==0:
                print(f'Epoch {i}: converged (0 mistakes)')  
                print(f'The model weights are : \n {self.weights}')
                break


    def predict(self,X):
        X = np.asarray(X)

        if X.ndim==1:
            X = X.reshape(1,-1)

        N  = len(X)

        if self.add_bias :
            X = np.column_stack((np.ones(N),X))

        scores = X @ self.weights.T 

        return self.unique_classes[np.argmax(scores,axis=1)]
       



## 1. Simple Linearly Separable Dataset

We construct a dataset with **3 classes** in 2D space such that they are linearly separable.

Each class occupies a distinct region.

The perceptron should learn weight vectors

$$
w_1, w_2, w_3
$$

such that

$$
\hat{y} = \arg\max_k w_k^T x
$$

correctly classifies all points.

---

## 2. Effect of Learning Rate Scaling

The perceptron update rule is

$$
w_{y_t} = w_{y_t} + \alpha x
$$

$$
w_{y_f} = w_{y_f} - \alpha x
$$

If we scale the learning rate $\alpha$, the **direction of weights remains the same**, but their **magnitude scales proportionally**.

Thus

$$
\alpha \uparrow \quad \Rightarrow \quad \|W\| \uparrow
$$

but the **decision boundary remains unchanged**.

---



In [2]:
import numpy as np

np.random.seed(42)

N = 100

X_class1 = np.random.randn(N,2) + np.array([3,3])
X_class2 = np.random.randn(N,2) + np.array([-3,3])
X_class3 = np.random.randn(N,2) + np.array([0,-3])

X_train = np.vstack([X_class1, X_class2, X_class3])

y_train = np.array([0]*N + [1]*N + [2]*N)

In [3]:
print("alpha = 0.1")

model_1 = MulticlassPerceptron(add_bias=True, epochs=1200, alpha=0.1)
model_1.fit(X_train , y_train)

pred_1 = model_1.predict(X_train)

print("\nTraining Accuracy :", np.mean(pred_1 == y_train))

alpha = 0.1
Epoch 20: converged (0 mistakes)
The model weights are : 
 [[-16.2         33.45909322  15.4993639 ]
 [  6.1        -33.31660743  15.95140543]
 [ 10.1         -0.14248579 -31.45076933]]

Training Accuracy : 1.0


In [4]:
print("alpha = 1.0")

model_2 = MulticlassPerceptron(add_bias=True, epochs=1200, alpha=1.0)
model_2.fit(X_train , y_train)

pred_2 = model_2.predict(X_train)

print("\nTraining Accuracy :", np.mean(pred_2 == y_train))

alpha = 1.0
Epoch 20: converged (0 mistakes)
The model weights are : 
 [[-162.          334.59093219  154.99363897]
 [  61.         -333.16607431  159.5140543 ]
 [ 101.           -1.42485788 -314.50769328]]

Training Accuracy : 1.0


## 3. Expected Behaviour

- Model should converge when data is linearly separable
- Increasing $\alpha$ scales weights proportionally
- Predictions remain identical despite scaling

---


In [5]:
norm_1 = np.linalg.norm(model_1.weights)
norm_2 = np.linalg.norm(model_2.weights)

print("Weight Scaling Comparison")

print("||W (alpha=0.1)|| :", norm_1)
print("||W (alpha=1.0)|| :", norm_2)

print("\nRatio (should be ~10):", norm_2 / norm_1)

Weight Scaling Comparison
||W (alpha=0.1)|| : 64.14839093621971
||W (alpha=1.0)|| : 641.4839093621964

Ratio (should be ~10): 9.99999999999999



## 4. Comparing Weight Scaling

We compare the norms of the weight matrices.

If the theory is correct:

$$
W_{\alpha=1} \approx 10 \times W_{\alpha=0.1}
$$

---

## 5. Prediction Consistency

Even though weights are scaled, predictions should remain identical.

$$
\arg\max_k w_k^T x
$$

is invariant to scaling of $W$.

In [6]:
print("Prediction Comparison")
same_predictions = np.all(pred_1 == pred_2)
print("Are predictions identical?", same_predictions)

Prediction Comparison
Are predictions identical? True
